# Grocery Demand Pulse
**Retail product demand estimation & stockout-risk analysis**

- Regression → how many units will each SKU sell tomorrow?
- Classification → will the SKU sell at all tomorrow?
- Data file: `grocery_sales_updated_datewise.csv` (keep it in the same folder as this notebook)

## 1. Settings & imports

In [ ]:
import json, os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score)

# ---------- SETTINGS (change here only) ----------
DATA_FILE     = "grocery_sales_updated_datewise.csv"
TEST_DAYS     = 4      # last 4 trading days are kept aside to test the model
MIN_SALE_DAYS = 3      # SKU must have sold on 3+ days to be modelled
CAP_QUANTILE  = 0.99   # cap extreme bulk orders
HORIZON_DAYS  = 7      # forecast window used for the stockout check later
FEATURES = ["yesterday", "two_days_ago", "avg_last_3_days", "avg_so_far", "weekday"]
OUT = "artifacts"
os.makedirs(OUT, exist_ok=True)

## 2. Load & clean the data

In [ ]:
df = pd.read_csv(DATA_FILE)
df.columns = ["date", "desc", "qty", "amount"]          # this file has no SKU code, so products are identified by description
df["desc"] = df["desc"].str.strip()
ids = {d: 1001 + i for i, d in enumerate(sorted(df["desc"].unique()))}   # our own numeric SKU id per product
df["sku"] = df["desc"].map(ids)
df["date"] = pd.to_datetime(df["date"])
print("Raw rows:", len(df))

returns = int((df["qty"] <= 0).sum())
df = df[df["qty"] > 0]                                   # remove returns
cap = df["qty"].quantile(CAP_QUANTILE)
df["qty"] = df["qty"].clip(upper=cap)                    # cap bulk orders
names = df.groupby("sku")["desc"].first()
print(f"Removed {returns} return row(s); capped orders at {cap:.0f} units")
df.head()

## 3. Daily table (SKU × day)
Days with no sale for a SKU are treated as zero demand. 9 Nov has no records at all, so it is treated as a non-trading day.

In [ ]:
wide = df.pivot_table(index="date", columns="sku", values="qty", aggfunc="sum").fillna(0)
n_days, last_date = len(wide), wide.index.max()
sale_days = (wide > 0).sum()
keep = sale_days[sale_days >= MIN_SALE_DAYS].index       # SKUs with enough history
print(f"{n_days} trading days, {wide.shape[1]} SKUs -> {len(keep)} modelled, "
      f"{wide.shape[1] - len(keep)} use simple average")

wide.sum(axis=1).plot(kind="bar", figsize=(9, 3.5), color="#2a9d8f", title="Total units sold per day")
plt.ylabel("units"); plt.tight_layout(); plt.show()

## 4. Build features & time-based split

In [ ]:
rows = []
for sku in keep:
    s = wide[sku]
    rows.append(pd.DataFrame({
        "date": s.index, "sku": sku, "day_no": range(n_days), "qty": s.values,
        "yesterday": s.shift(1).values,
        "two_days_ago": s.shift(2).values,
        "avg_last_3_days": s.shift(1).rolling(3).mean().values,
        "avg_so_far": s.shift(1).expanding().mean().values,
        "weekday": s.index.dayofweek}))
data = pd.concat(rows).dropna().reset_index(drop=True)
data["sold"] = (data["qty"] > 0).astype(int)             # classification target

split = n_days - TEST_DAYS                               # no shuffling: test = later days
train, test = data[data.day_no < split].copy(), data[data.day_no >= split].copy()
print(f"Train rows: {len(train)} | Test rows: {len(test)} (test starts {test.date.min().date()})")
data.head()

## 5. Regression — how many units?

In [ ]:
def score(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p)))}

reg_models = {"Linear Regression": LinearRegression(),
              "Decision Tree (depth 3)": DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=1)}
results = {"Baseline: same as yesterday": score(test.qty, test.yesterday)}
preds = {}
for name, m in reg_models.items():
    m.fit(train[FEATURES], train.qty)
    preds[name] = np.clip(m.predict(test[FEATURES]), 0, None)
    results[name] = score(test.qty, preds[name])

reg_table = pd.DataFrame(results).T.sort_values("RMSE")
best_name = min(reg_models, key=lambda n: results[n]["RMSE"])   # RMSE: demand has many zero days
reg_table.to_csv(f"{OUT}/model_comparison.csv")
print("Best regression model:", best_name)
reg_table.round(3)

## 6. Classification — will it sell?

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(train[FEATURES], train.sold)
test["sale_prob"] = clf.predict_proba(test[FEATURES])[:, 1]
test_class = (test.sale_prob >= 0.5).astype(int)

clf_scores = {"Accuracy": accuracy_score(test.sold, test_class),
              "Precision": precision_score(test.sold, test_class, zero_division=0),
              "Recall": recall_score(test.sold, test_class, zero_division=0),
              "F1": f1_score(test.sold, test_class, zero_division=0),
              "Baseline accuracy (always 'no sale')": 1 - test.sold.mean()}
pd.Series(clf_scores, name="value").to_csv(f"{OUT}/classifier_scores.csv")
pd.Series(clf_scores, name="value").round(3).to_frame()

## 7. Forecast-error analysis

In [ ]:
test["predicted"] = preds[best_name]
test["error"] = test.predicted - test.qty
test["desc"] = test.sku.map(names)
test[["date", "sku", "desc", "qty", "predicted", "error", "sale_prob"]].to_csv(f"{OUT}/test_predictions.csv", index=False)

by_day = test.groupby("date").agg(actual=("qty", "sum"), predicted=("predicted", "sum"))
by_day.to_csv(f"{OUT}/daily_error_analysis.csv")
by_day.plot(kind="bar", figsize=(7, 3.5), color=["#264653", "#e9c46a"], title="Actual vs predicted units (test days)")
plt.tight_layout(); plt.show()

per_sku = test.groupby("sku").agg(actual_total=("qty", "sum"), predicted_total=("predicted", "sum"),
                                  avg_abs_error=("error", lambda e: e.abs().mean())).reset_index()
per_sku["desc"] = per_sku.sku.map(names)
per_sku["pct_error"] = (per_sku.predicted_total - per_sku.actual_total) / per_sku.actual_total.replace(0, np.nan)
per_sku.to_csv(f"{OUT}/sku_error_analysis.csv", index=False)
resid_std = test.groupby("sku")["error"].std()
per_sku.sort_values("avg_abs_error", ascending=False).head(10).round(2)

## 8. Retrain on all data & forecast the next trading day

In [ ]:
final_reg = reg_models[best_name].fit(data[FEATURES], data.qty)
final_clf = clf.fit(data[FEATURES], data.sold)

next_day = last_date + pd.Timedelta(days=1)
nxt = pd.DataFrame({
    "yesterday": wide[keep].iloc[-1], "two_days_ago": wide[keep].iloc[-2],
    "avg_last_3_days": wide[keep].iloc[-3:].mean(), "avg_so_far": wide[keep].mean(),
    "weekday": next_day.dayofweek})
nxt_pred = pd.Series(np.clip(final_reg.predict(nxt[FEATURES]), 0, None), index=nxt.index)
nxt_prob = pd.Series(final_clf.predict_proba(nxt[FEATURES])[:, 1], index=nxt.index)

fc = pd.DataFrame({"sku": wide.columns})
fc["desc"] = fc.sku.map(names)
fc["sales_days"] = fc.sku.map(sale_days)
fc["total_qty"] = fc.sku.map(wide.sum())
fc["avg_daily"] = fc.sku.map(wide.mean())
fc["daily_std"] = fc.sku.map(wide.std()).fillna(0)
fc["method"] = np.where(fc.sku.isin(keep), best_name, "Average daily sales")
fc["forecast_next_day"] = fc.sku.map(nxt_pred).fillna(fc.avg_daily)
fc["sale_probability"] = fc.sku.map(nxt_prob).fillna(fc.sales_days / n_days)
fc["resid_std"] = fc.sku.map(resid_std).fillna(fc.daily_std)
fc["forecast_horizon"] = fc.forecast_next_day * HORIZON_DAYS
fc["forecast_date"] = next_day.date()
fc = fc.sort_values("forecast_horizon", ascending=False)
fc.to_csv(f"{OUT}/sku_forecasts.csv", index=False)

top = fc.head(10).iloc[::-1]
plt.figure(figsize=(8, 4.5))
plt.barh(top.desc, top.forecast_next_day, color="#2a9d8f")
plt.title(f"Top 10 SKUs - expected units on {next_day.date()}"); plt.xlabel("units")
plt.tight_layout(); plt.show()
fc[["desc", "method", "forecast_next_day", "sale_probability", "forecast_horizon"]].head(10).round(2)

## 9. Save the model

In [ ]:
wide.reset_index().melt(id_vars="date", var_name="sku", value_name="qty").to_csv(f"{OUT}/history.csv", index=False)
joblib.dump({"regressor": final_reg, "classifier": final_clf, "features": FEATURES,
             "best_regressor": best_name, "last_date": str(last_date.date()),
             "horizon_days": HORIZON_DAYS, "forecasts": fc}, "demand_model.pkl")
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump({"best_regressor": best_name, "regression": results, "classification": clf_scores}, f, indent=2)
print("Model pipeline saved as demand_model.pkl")